# LC 207 — Course Schedule
**Day 42 | Graphs: Topological Sort | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A valid course schedule exists if and
only if the prerequisite graph contains <em>no cycle</em>. Use
DFS with a three-state colour array (unvisited / visiting / done)
to detect a back-edge in O(V+E) time.
</div>

## Official Problem Statement

There are `numCourses` courses labelled `0` to `numCourses - 1`.
You are given an array `prerequisites` where
`prerequisites[i] = [a, b]` means you must take course `b`
**before** course `a`.

Return `true` if you can finish all courses, `false` otherwise.

**Constraints**
- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= 5000`
- `prerequisites[i].length == 2`
- `0 <= a, b < numCourses`
- All pairs `[a, b]` are **unique**

**Examples**
```
Input:  numCourses=2, prerequisites=[[1,0]]
Output: True   # take 0 then 1

Input:  numCourses=2, prerequisites=[[1,0],[0,1]]
Output: False  # 0 needs 1 and 1 needs 0 — cycle
```

## Walk Through an Example by Hand

```
numCourses = 4
prerequisites = [[1,0],[2,1],[3,2],[1,3]]  # cycle: 1->3->2->1

Adjacency list (a depends on b  =>  edge b->a):
  0: [1]
  1: [2]
  2: [3]
  3: [1]   <-- back to 1

DFS from node 0:
  visit(0): state[0]=1
    visit(1): state[1]=1
      visit(2): state[2]=1
        visit(3): state[3]=1
          neighbour 1 -> state[1]==1  CYCLE FOUND -> return True

has_cycle=True -> canFinish returns False
```

Clean example (no cycle):
```
numCourses=3, prerequisites=[[1,0],[2,1]]
DFS(0)->DFS(1)->DFS(2): all neighbours done, no back-edge
state: [2,2,2]  -> return True
```

## What This Is Actually Asking

We are asked whether a directed graph has a cycle.
Each course is a node; each prerequisite pair is a directed edge.
If any cycle exists, it is impossible to satisfy all prerequisites,
so we cannot finish every course.
The three-colour DFS gives us cycle detection in a single pass
without any extra data structures beyond the state array.

## The Picture

```
Graph (no cycle)          Graph (cycle)

  0 --> 1 --> 3           0 --> 1 --> 3
        |                       ^     |
        v                       |     v
        2                       +---- 2

DFS state legend:
  0 = WHITE  (unvisited)
  1 = GRAY   (on current DFS stack — visiting)
  2 = BLACK  (fully explored — done)

Cycle detection rule:
  If DFS reaches a GRAY node -> back-edge -> CYCLE

Step-by-step (cycle graph, start node 0):

  Call      state before   action
  ------    ------------   ---------------------
  dfs(0)    [0,0,0,0]      mark 0 GRAY -> [1,0,0,0]
  dfs(1)    [1,0,0,0]      mark 1 GRAY -> [1,1,0,0]
  dfs(3)    [1,1,0,0]      mark 3 GRAY -> [1,1,0,1]
  dfs(2)    [1,1,0,1]      mark 2 GRAY -> [1,1,1,1]
    nbr=1 state==GRAY      CYCLE! return True immediately

No cycle path — node finishes all neighbours:
  dfs(3): no neighbours -> mark BLACK -> [*,*,*,2]
  dfs(1): done         -> mark BLACK -> [*,2,*,2]
  dfs(0): done         -> mark BLACK -> [2,2,*,*]
```

## When To Use This Pattern

- When tasks/jobs have dependencies and you need to know if a valid
  ordering exists, think **cycle detection via DFS**.
- When a graph is directed and you must detect a back-edge,
  think **three-colour (white/gray/black) DFS**.
- When nodes can be disconnected, think **outer loop over all
  nodes** calling DFS only on unvisited ones.
- When the problem says "can you complete all X", think
  **is the dependency graph a DAG?**
- When constraints are up to 2000 nodes / 5000 edges,
  think **O(V+E) DFS is safe**.

## The Approach

Build an adjacency list from the prerequisites array, treating
each pair `[a, b]` as a directed edge from `b` to `a`.
Maintain a `state` array of length `numCourses` initialised to 0.
Run DFS from every unvisited node; inside DFS, mark the current
node as GRAY (1) before recursing into neighbours, then mark it
BLACK (2) when fully done — if we ever reach a GRAY neighbour
we found a cycle and immediately return `False`.

In [ ]:
from typing import List
from collections import defaultdict, deque

In [ ]:
def test_harness(func):
    """Run test cases for LC 207 canFinish."""
    cases = [
        # (numCourses, prerequisites, expected)
        (2, [[1, 0]], True),
        (2, [[1, 0], [0, 1]], False),
        (1, [], True),
        (4, [[1, 0], [2, 1], [3, 2]], True),
        (4, [[1, 0], [2, 1], [3, 2], [1, 3]], False),
        (3, [[0, 1], [0, 2], [1, 2]], True),
        (5, [[1,0],[2,1],[3,2],[4,3],[0,4]], False),
    ]
    passed = 0
    for i, (n, prereqs, expected) in enumerate(cases):
        result = func(n, prereqs)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"n={n} prereqs={prereqs} "
            f"-> {result} (expected {expected})"
        )
    print(f"\nResult: {passed}/{len(cases)} passed")

In [ ]:
def canFinish(
    numCourses: int,
    prerequisites: List[List[int]]
) -> bool:
    """
    Determine if all courses can be finished.

    Uses DFS cycle detection with three states:
      0 = unvisited (WHITE)
      1 = visiting  (GRAY)  — on current DFS stack
      2 = done      (BLACK) — fully explored

    If a GRAY node is reached during DFS, a cycle exists
    and the courses cannot all be completed.

    Args:
        numCourses:    total number of courses (0..numCourses-1)
        prerequisites: list of [a, b] meaning b must come before a

    Returns:
        True  if no cycle exists (valid schedule possible)
        False if a cycle exists

    Time:  O(V + E)
    Space: O(V + E)
    """
    # --- build adjacency list ---
    print(f"[DEBUG] numCourses={numCourses}, "
          f"prerequisites={prerequisites}")

    graph = defaultdict(list)
    for a, b in prerequisites:
        graph[b].append(a)  # b must come before a -> edge b->a

    print(f"[DEBUG] graph={dict(graph)}")

    state = [0] * numCourses  # 0=WHITE,1=GRAY,2=BLACK

    def dfs(node: int) -> bool:
        """Return True if a cycle is found from this node."""
        if state[node] == 1:  # GRAY -> back-edge -> cycle
            print(f"[DEBUG] cycle detected at node {node}")
            return True
        if state[node] == 2:  # BLACK -> already safe
            return False

        state[node] = 1       # mark GRAY
        for neighbour in graph[node]:
            if dfs(neighbour):
                return True
        state[node] = 2       # mark BLACK
        print(f"[DEBUG] node {node} fully explored (BLACK)")
        return False

    for course in range(numCourses):
        if state[course] == 0:
            if dfs(course):
                return False  # cycle found

    pass  # replace with: return True

In [ ]:
# Uncomment and run when solution is ready
# test_harness(canFinish)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (try all orderings) | O(V!) | O(V) | Infeasible |
| DFS cycle detection (optimal) | O(V+E) | O(V+E) | One DFS pass |
| BFS / Kahn's algorithm | O(V+E) | O(V+E) | Alt. approach |

V = numCourses, E = len(prerequisites)

## Real World Connection

At **Citi**, financial products often have layered dependencies —
a derivative instrument may require its underlying data feeds to
be loaded first. A circular dependency in the load order would
deadlock the entire pipeline, so cycle detection is run during
deployment validation. On **AWS Glue**, before scheduling a DAG
of ETL jobs, the orchestrator checks for cycles in the dependency
graph to prevent infinite waits. As a **Data Engineer**, catching
circular references early — whether in dbt model dependencies,
Airflow DAGs, or Spark job chains — is a core operational skill
that prevents costly production incidents.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra